In [4]:
from model.ctabgan import CTABGAN
from model.eval.evaluation import get_utility_metrics, stat_sim, privacy_metrics
import numpy as np
import pandas as pd
import glob

In [5]:
num_exp = 5
dataset = "WiDS"
real_path = "Real_Datasets/WiDS_clean.csv"
fake_file_root = "Fake_Datasets"

In [4]:
synthesizer = CTABGAN(
    raw_csv_path=real_path,
    test_ratio=0.20,

    categorical_columns=[
    "elective_surgery",
    "ethnicity",
    "gender",
    "icu_admit_source",
    "icu_stay_type",
    "icu_type",
    "readmission_status",
    "apache_post_operative",

    # binary flags
    "arf_apache",
    "gcs_unable_apache",
    "intubated_apache",
    "ventilated_apache",
    "aids",
    "cirrhosis",
    "diabetes_mellitus",
    "hepatic_failure",
    "immunosuppression",
    "leukemia",
    "lymphoma",
    "solid_tumor_with_metastasis",

    # IMPORTANT: diagnosis codes
    "apache_2_diagnosis",
    "apache_3j_diagnosis",

    # other categorical
    "apache_3j_bodysystem",
    "apache_2_bodysystem",

    "hospital_death"
],

    log_columns=[],

    mixed_columns={},

    general_columns=[
        "height",
        "pre_icu_los_days",
        "heart_rate_apache",
        "map_apache",
        "resprate_apache",
        "d1_diasbp_max",
        "d1_diasbp_min",
        "d1_diasbp_noninvasive_max",
        "d1_diasbp_noninvasive_min",
        "d1_heartrate_max",
        "d1_heartrate_min",
        "d1_mbp_max",
        "d1_mbp_min",
        "d1_mbp_noninvasive_max",
        "d1_mbp_noninvasive_min",
        "d1_resprate_max",
        "d1_resprate_min",
        "d1_spo2_max",
        "d1_spo2_min",
        "d1_sysbp_max",
        "d1_sysbp_min",
        "d1_sysbp_noninvasive_max",
        "d1_sysbp_noninvasive_min"
    ],

    non_categorical_columns=[],
    integer_columns=[],

    problem_type={"Classification": "hospital_death"},

    class_dim=(256, 256),
    random_dim=100,
    num_channels=64,
    l2scale=1e-5,
    batch_size=512,
    epochs=150
)

for i in range(num_exp):
    synthesizer.fit()
    syn = synthesizer.generate_samples()
    syn.to_csv(
        fake_file_root + "/" + dataset + "/" + dataset + "_fake_{exp}.csv".format(exp=i),
        index=False
    )

100%|██████████| 150/150 [07:45<00:00,  3.10s/it]


Finished training in 469.0533401966095  seconds.


100%|██████████| 150/150 [07:35<00:00,  3.04s/it]


Finished training in 458.2635180950165  seconds.


100%|██████████| 150/150 [07:35<00:00,  3.04s/it]


Finished training in 458.29652404785156  seconds.


100%|██████████| 150/150 [07:32<00:00,  3.02s/it]


Finished training in 455.13194704055786  seconds.


100%|██████████| 150/150 [07:30<00:00,  3.00s/it]


Finished training in 452.91281270980835  seconds.


In [6]:
fake_paths = glob.glob(fake_file_root+"/"+dataset+"/"+"*")

In [7]:
import pandas as pd
df = pd.read_csv(fake_file_root+"/"+dataset+"/"+ dataset+"_fake_0.csv")
df.info()
df.head()
df.tail()
df.isna().sum()
df.replace([np.inf, -np.inf], np.nan).isna().sum()

FileNotFoundError: [Errno 2] No such file or directory: 'Fake_Datasets/WiDS/WiDS_fake_0.csv'

In [22]:
import pandas as pd
df = pd.read_csv(fake_file_root+"/"+dataset+"/"+ dataset+"_fakeU_0.csv")
df.info()
df.head()
df.tail()
df.isna().sum()
df.replace([np.inf, -np.inf], np.nan).isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23698 entries, 0 to 23697
Data columns (total 32 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age_at_intime         23698 non-null  int64  
 1   heart_rate_min        23698 non-null  float64
 2   heart_rate_max        23698 non-null  float64
 3   heart_rate_mean       23698 non-null  float64
 4   sbp_min               23698 non-null  float64
 5   sbp_max               23698 non-null  float64
 6   sbp_mean              23698 non-null  float64
 7   dbp_min               23698 non-null  float64
 8   dbp_max               23698 non-null  float64
 9   dbp_mean              23698 non-null  float64
 10  resp_rate_min         23698 non-null  float64
 11  resp_rate_max         23698 non-null  float64
 12  resp_rate_mean        23698 non-null  float64
 13  temperature_min       23698 non-null  float64
 14  temperature_max       23698 non-null  float64
 15  temperature_mean   

age_at_intime           0
heart_rate_min          0
heart_rate_max          0
heart_rate_mean         0
sbp_min                 0
sbp_max                 0
sbp_mean                0
dbp_min                 0
dbp_max                 0
dbp_mean                0
resp_rate_min           0
resp_rate_max           0
resp_rate_mean          0
temperature_min         0
temperature_max         0
temperature_mean        0
spo2_min                0
spo2_max                0
spo2_mean               0
creatinine_min          0
creatinine_max          0
sodium_min              0
sodium_max              0
potassium_min           0
potassium_max           0
hemoglobin_min          0
hemoglobin_max          0
wbc_min                 0
wbc_max                 0
gender_F                0
gender_M                0
hospital_expire_flag    0
dtype: int64

In [16]:
def make_numeric_for_utility(path_in, path_out, target="hospital_expire_flag"):
    df = pd.read_csv(path_in)

    # Ensure target is last or separated (depending on your eval code)
    y = df[target]
    X = df.drop(columns=[target])

    # One-hot encode any object/categorical columns (e.g., gender)
    X_enc = pd.get_dummies(X, drop_first=False)

    out = pd.concat([X_enc, y], axis=1)
    out.to_csv(path_out, index=False)
    return path_out

real_path_u = make_numeric_for_utility(real_path, "Real_Datasets/MimicIVU.csv")

fake_paths_u = []
for i, fp in enumerate(fake_paths):
    fake_paths_u.append(make_numeric_for_utility(fp, f"Fake_Datasets/MimicIV/MimicIV_fakeU_{i}.csv"))

    # after generating real one-hot columns, align fake to real:
real_cols = pd.read_csv(real_path_u).columns

def align_to_real(path_in, real_cols, target):
    df = pd.read_csv(path_in)
    y = df[target]
    X = pd.get_dummies(df.drop(columns=[target]), drop_first=False)

    X = X.reindex(columns=[c for c in real_cols if c != target], fill_value=0)
    out = pd.concat([X, y], axis=1)
    return out

In [17]:
def nan_overview(path, name="dataset"):
    print(f"\n{'='*60}")
    print(f"NaN overview for: {name}")
    print(f"{'='*60}")
    
    df = pd.read_csv(path)
    total_rows = len(df)
    
    # Replace inf with NaN so we catch those too
    df = df.replace([np.inf, -np.inf], np.nan)
    
    # Column-wise NaN count
    nan_counts = df.isna().sum()
    nan_percent = (nan_counts / total_rows * 100).round(2)
    
    summary = pd.DataFrame({
        "NaN Count": nan_counts,
        "NaN %": nan_percent
    }).sort_values("NaN Count", ascending=False)
    
    print("\nColumn-wise NaN summary:")
    display(summary[summary["NaN Count"] > 0])
    
    # Row-level info
    rows_with_nan = df.isna().any(axis=1).sum()
    print(f"\nTotal rows: {total_rows}")
    print(f"Rows containing at least one NaN: {rows_with_nan}")
    print(f"Percentage of rows with NaN: {(rows_with_nan/total_rows*100):.2f}%")
    
    # Show example problematic rows
    if rows_with_nan > 0:
        print("\nExample rows with NaNs:")
        display(df[df.isna().any(axis=1)].head())
    
    return summary


# === RUN ON REAL DATA ===
real_summary = nan_overview(real_path, name="REAL DATA")


# === RUN ON FAKE DATASETS ===
fake_summaries = {}
for i, fp in enumerate(fake_paths):
    fake_summaries[fp] = nan_overview(fp, name=f"FAKE DATA {i}")


NaN overview for: REAL DATA

Column-wise NaN summary:


,NaN Count,NaN %



Total rows: 23698
Rows containing at least one NaN: 0
Percentage of rows with NaN: 0.00%

NaN overview for: FAKE DATA 0

Column-wise NaN summary:


,NaN Count,NaN %



Total rows: 23698
Rows containing at least one NaN: 0
Percentage of rows with NaN: 0.00%

NaN overview for: FAKE DATA 1

Column-wise NaN summary:


...


Total rows: 23698
Rows containing at least one NaN: 0
Percentage of rows with NaN: 0.00%

NaN overview for: FAKE DATA 13

Column-wise NaN summary:


,NaN Count,NaN %



Total rows: 23698
Rows containing at least one NaN: 0
Percentage of rows with NaN: 0.00%

NaN overview for: FAKE DATA 14

Column-wise NaN summary:


,NaN Count,NaN %



Total rows: 23698
Rows containing at least one NaN: 0
Percentage of rows with NaN: 0.00%


In [18]:
model_dict =  {"Classification":["lr","dt","rf","mlp","svm"]}
result_mat = get_utility_metrics(real_path_u,fake_paths_u,"MinMax",model_dict, test_ratio = 0.20)

result_df  = pd.DataFrame(result_mat,columns=["Acc","AUC","F1_Score"])
result_df.index = list(model_dict.values())[0]
result_df

ValueError: Unknown label type: unknown. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [23]:
mimic_categorical = ['hospital_death']
stat_res_avg = []
for fake_path in fake_paths:
    stat_res = stat_sim(real_path,fake_path,mimic_categorical)
    stat_res_avg.append(stat_res)

stat_columns = ["Average WD (Continuous Columns","Average JSD (Categorical Columns)","Correlation Distance"]
stat_results = pd.DataFrame(np.array(stat_res_avg).mean(axis=0).reshape(1,3),columns=stat_columns)
stat_results

TypeError: unsupported operand type(s) for +: 'float' and 'str'

In [20]:
priv_res_avg = []
for fake_path in fake_paths_u:
    priv_res = privacy_metrics(real_path_u,fake_path)
    priv_res_avg.append(priv_res)
    
privacy_columns = ["DCR between Real and Fake (5th perc)","DCR within Real(5th perc)","DCR within Fake (5th perc)","NNDR between Real and Fake (5th perc)","NNDR within Real (5th perc)","NNDR within Fake (5th perc)"]
privacy_results = pd.DataFrame(np.array(priv_res_avg).mean(axis=0).reshape(1,6),columns=privacy_columns)
privacy_results

,DCR between Real and Fake (5th perc),DCR within Real(5th perc),DCR within Fake (5th perc),NNDR between Real and Fake (5th perc),NNDR within Real (5th perc),NNDR within Fake (5th perc)
0,2.520837,2.219882,2.303182,0.840909,0.817386,0.806592


In [24]:
import os
import pandas as pd

from Evaluation.cdf_tail_metrics import CDFTailMetrics
from Evaluation.support_coverage import SupportCoverage
from Evaluation.rare_event_recall import RareEventRecall

n_experiments = 5

cdf_eval = CDFTailMetrics(label_col="hospital_death", tau=0.9)
sc_eval  = SupportCoverage(label_col="hospital_death",  k=2,  # pairwise support coverage
    n_bins=10,
    rare_threshold=0.05,
    max_subsets=100,  # sample up to 100 feature pairs
    include_label_in_combo=True,
    random_state=42)
rer_eval = RareEventRecall(label_col="hospital_death")

rows = []

for exp in range(n_experiments):
    syn_path = os.path.join(fake_file_root, dataset, f"{dataset}_fakeU_{exp}.csv")
    if not os.path.exists(syn_path):
        print("Missing:", syn_path)
        continue

    res = {"exp": exp, "syn_path": syn_path}
    res.update(cdf_eval.evaluate_paths(real_path_u, syn_path))
    res.update(sc_eval.evaluate_paths(real_path_u, syn_path))
    res.update(rer_eval.evaluate_paths(real_path_u, syn_path))
    rows.append(res)

results_df = pd.DataFrame(rows)
results_df


TypeError: numpy boolean subtract, the `-` operator, is not supported, use the bitwise_xor, the `^` operator, or the logical_xor function instead.